# Build Metadata Store — data.wa.gov

Fetches, for every `data.wa.gov` dataset listed in `DatasetsWithSolidMetadata - Sheet1.csv`:

- **Dataset name** and **description**
- Every **column name**, its **type**, and its **column description**
- Up to **10 distinct sample values per column**

…and stores it so you can fetch it later:

- `metadata_store/<uid>.json` — one file per dataset (resumable; re-running skips ones already saved)
- `all_metadata.json` — a single combined `{uid: record}` file
- `metadata_store/_index.json` — run summary (per-dataset status + any errors)

Pure standard library (`csv`, `json`, `urllib`) — nothing to install. Data comes from the Socrata APIs:
`/api/views/<uid>.json` (name, description, per-column descriptions) and `/resource/<uid>.json` (sample rows).

## 1. Configuration

In [1]:
import csv
import json
import re
import time
import urllib.request
import urllib.error
from datetime import datetime, timezone
from pathlib import Path

# ---- Inputs / outputs ----
CSV_PATH = Path("DatasetsWithSolidMetadata - Sheet1.csv")
DOMAIN = "data.wa.gov"  # only fetch datasets on this domain
OUT_DIR = Path("metadata_store")  # per-dataset JSON files land here
COMBINED_PATH = Path("all_metadata.json")
INDEX_PATH = OUT_DIR / "_index.json"

# ---- Fetch behavior ----
SAMPLES_PER_COLUMN = 10  # distinct non-null example values to keep per column
ROWS_TO_SCAN = 100  # rows pulled per dataset to harvest those samples
REQUEST_TIMEOUT = 60  # seconds per HTTP request
SLEEP_BETWEEN = 0.3  # politeness pause between datasets (seconds)
APP_TOKEN = (
    ""  # optional Socrata app token (raises rate limits); leave "" for anonymous
)
SKIP_COMPUTED = True  # drop Socrata system/computed columns (fieldName starts with ":")
OVERWRITE = False  # False = resume (skip datasets already saved); True = re-fetch all

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUT_DIR.resolve()}")

Output dir: /home/wynterlin/Documents/GitHub/MetadataLearn/metadata_store


## 2. Load the dataset list from the CSV

De-duplicates UIDs and keeps only rows on the target domain.

In [2]:
def load_uids(csv_path, domain):
    seen, out = set(), []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        for row in csv.DictReader(f):
            uid = (row.get("UID") or "").strip()
            dom = (row.get("Domain") or "").strip()
            if not uid or uid in seen:
                continue
            if domain and dom and dom != domain:
                continue
            seen.add(uid)
            out.append(
                {
                    "uid": uid,
                    "name": (row.get("Name") or "").strip(),
                    "url": (row.get("url") or "").strip(),
                }
            )
    return out


datasets = load_uids(CSV_PATH, DOMAIN)
print(f"{len(datasets)} unique {DOMAIN} datasets to fetch")
for d in datasets[:5]:
    print(f"  {d['uid']}  {d['name'][:60]}")
print("  ...")

101 unique data.wa.gov datasets to fetch
  r9b2-b8ff  WA Ground Ambulance Locally Set Rates
  fysr-7kwx  Health Plan Prior Authorization Data
  gi9j-78eu  Washington State Insurers Dental Loss Ratios
  bxeh-ranj  Annual County Road Related Expenditures
  s8fv-s6fc  Annual Aggregated County Road Mileage
  ...


## 3. Helper functions

In [3]:
def _request(url):
    """GET a URL and parse JSON, sending the app token if configured."""
    req = urllib.request.Request(url)
    if APP_TOKEN:
        req.add_header("X-App-Token", APP_TOKEN)
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
        return json.load(resp)


def strip_html(s):
    """Remove HTML tags and collapse whitespace in a metadata string."""
    if not s:
        return ""
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def epoch_to_iso(ts):
    """Convert a Socrata epoch-seconds timestamp to ISO 8601 (UTC)."""
    if not ts:
        return None
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).isoformat()
    except (ValueError, TypeError, OSError):
        return ts


def is_real_column(col):
    """True for user-facing data columns; skips system/computed (':' / ':@computed_*')."""
    field = col.get("fieldName") or ""
    if not field:
        return False
    if SKIP_COMPUTED and field.startswith(":"):
        return False
    return True


def _sample_value(v):
    """Make a row value JSON-friendly (location/point cells arrive as dicts)."""
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False, default=str)
    return v


def collect_samples(field, rows, cached_top, k):
    """Up to k distinct non-null values for `field`, from rows then cachedContents.top."""
    samples, seen = [], set()

    def add(raw):
        if raw is None or raw == "":
            return False
        val = _sample_value(raw)
        key = val if isinstance(val, str) else json.dumps(val, default=str)
        if key in seen:
            return False
        seen.add(key)
        samples.append(val)
        return True

    for r in rows:
        if field in r and add(r[field]) and len(samples) >= k:
            return samples
    for t in cached_top or []:  # fallback when rows didn't yield enough
        if add(t.get("item")) and len(samples) >= k:
            break
    return samples

In [4]:
def fetch_view(uid):
    """Dataset-level metadata: name, description, and column definitions."""
    return _request(f"https://{DOMAIN}/api/views/{uid}.json")


def fetch_rows(uid, limit):
    """Up to `limit` sample rows (keyed by column fieldName)."""
    return _request(f"https://{DOMAIN}/resource/{uid}.json?$limit={limit}")


def build_record(uid, fallback_name=""):
    """Assemble the stored record for one dataset."""
    view = fetch_view(uid)

    row_error = None
    try:
        rows = fetch_rows(uid, ROWS_TO_SCAN)
    except Exception as e:  # keep the dataset even if row fetch fails
        rows, row_error = [], f"{type(e).__name__}: {e}"

    columns = []
    for col in view.get("columns", []) or []:
        if not is_real_column(col):
            continue
        field = col.get("fieldName")
        cached_top = (col.get("cachedContents") or {}).get("top")
        columns.append(
            {
                "name": col.get("name"),
                "field": field,
                "type": col.get("dataTypeName"),
                "description": strip_html(col.get("description")),
                "samples": collect_samples(field, rows, cached_top, SAMPLES_PER_COLUMN),
            }
        )

    meta = view.get("metadata") or {}
    record = {
        "uid": uid,
        "name": view.get("name") or fallback_name,
        "description": strip_html(view.get("description")),
        "attribution": view.get("attribution"),
        "category": view.get("category"),
        "tags": view.get("tags") or [],
        "row_label": meta.get("rowLabel") if isinstance(meta, dict) else None,
        "column_count": len(columns),
        "created_at": epoch_to_iso(view.get("createdAt")),
        "updated_at": epoch_to_iso(
            view.get("rowsUpdatedAt") or view.get("viewLastModified")
        ),
        "domain": DOMAIN,
        "page_url": f"https://{DOMAIN}/d/{uid}",
        "api_endpoint": f"https://{DOMAIN}/resource/{uid}.json",
        "rows_scanned": len(rows),
        "columns": columns,
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }
    if row_error:
        record["row_fetch_error"] = row_error
    return record

## 4. Fetch every dataset

Writes one JSON file per dataset as it goes. Safe to re-run — already-saved datasets are skipped unless `OVERWRITE = True`. Transient network errors are caught and recorded so one bad dataset doesn't halt the run.

In [5]:
index, errors = [], []

for i, d in enumerate(datasets, 1):
    uid = d["uid"]
    out_path = OUT_DIR / f"{uid}.json"

    if out_path.exists() and not OVERWRITE:
        rec = json.loads(out_path.read_text(encoding="utf-8"))
        index.append(
            {
                "uid": uid,
                "name": rec.get("name"),
                "column_count": rec.get("column_count"),
                "status": "cached",
            }
        )
        print(f"[{i:>3}/{len(datasets)}] {uid}  cached")
        continue

    try:
        rec = build_record(uid, d["name"])
        out_path.write_text(
            json.dumps(rec, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
        )
        index.append(
            {
                "uid": uid,
                "name": rec["name"],
                "column_count": rec["column_count"],
                "rows_scanned": rec["rows_scanned"],
                "status": "ok",
            }
        )
        note = "  (row fetch failed)" if rec.get("row_fetch_error") else ""
        print(
            f"[{i:>3}/{len(datasets)}] {uid}  ok  {rec['column_count']:>2} cols, "
            f"{rec['rows_scanned']:>3} rows  {(rec['name'] or '')[:46]}{note}"
        )
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        errors.append({"uid": uid, "error": msg})
        index.append({"uid": uid, "name": d["name"], "status": "error"})
        print(f"[{i:>3}/{len(datasets)}] {uid}  ERROR  {msg}")

    time.sleep(SLEEP_BETWEEN)

n_ok = sum(1 for x in index if x["status"] == "ok")
n_cached = sum(1 for x in index if x["status"] == "cached")
print(f"\nDone. ok={n_ok}  cached={n_cached}  errors={len(errors)}")

[  1/101] r9b2-b8ff  ok  18 cols, 100 rows  WA Ground Ambulance Locally Set Rates


[  2/101] fysr-7kwx  ok  19 cols, 100 rows  Health Plan Prior Authorization Data


[  3/101] gi9j-78eu  ok  13 cols, 100 rows  Washington State Insurers Dental Loss Ratios


[  4/101] bxeh-ranj  ok  14 cols, 100 rows  Annual County Road Related Expenditures


[  5/101] s8fv-s6fc  ok  13 cols, 100 rows  Annual Aggregated County Road Mileage


...

[ 98/101] pzcu-jpab  ok   9 cols, 100 rows  Washington State CPA (Certified Public Account


[ 99/101] brpd-b6zd  ok  31 cols, 100 rows  Washington State Liquor and Cannabis Board (LC


[100/101] 9dee-kzm5  ok  31 cols, 100 rows  Washington State Liquor and Cannabis Board (LC


[101/101] vgcw-qfjm  ok  38 cols,  16 rows  Washington State Liquor and Cannabis Board (LC



Done. ok=101  cached=0  errors=0


## 5. Write the combined file + run index

In [6]:
records = {}
for p in sorted(OUT_DIR.glob("*.json")):
    if p.name == "_index.json":
        continue
    records[p.stem] = json.loads(p.read_text(encoding="utf-8"))

COMBINED_PATH.write_text(
    json.dumps(records, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)

INDEX_PATH.write_text(
    json.dumps(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "domain": DOMAIN,
            "source_csv": str(CSV_PATH),
            "samples_per_column": SAMPLES_PER_COLUMN,
            "rows_scanned_per_dataset": ROWS_TO_SCAN,
            "count": len(records),
            "datasets": index,
            "errors": errors,
        },
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print(f"Wrote {COMBINED_PATH}  ({len(records)} datasets)")
print(f"Wrote {INDEX_PATH}")
if errors:
    print(f"\n{len(errors)} dataset(s) errored — see _index.json:")
    for e in errors:
        print(f"  {e['uid']}: {e['error']}")

Wrote all_metadata.json  (101 datasets)
Wrote metadata_store/_index.json


## 6. Fetch the data later

Two convenience loaders. `load_dataset` reads a single dataset's JSON; `load_all` reads the combined file.

In [7]:
def load_dataset(uid, store_dir=OUT_DIR):
    """Load one dataset's metadata record by UID."""
    return json.loads((Path(store_dir) / f"{uid}.json").read_text(encoding="utf-8"))


def load_all(path=COMBINED_PATH):
    """Load every dataset record as a {uid: record} dict."""
    return json.loads(Path(path).read_text(encoding="utf-8"))


# --- quick verification of one dataset ---
rec = load_dataset("r9b2-b8ff")
print(f"{rec['name']}  ({rec['column_count']} columns)")
print(f"description: {rec['description'][:120]}...\n")
for c in rec["columns"][:4]:
    print(f"• {c['name']}  [{c['type']}]")
    print(f"    desc:    {(c['description'] or '(none)')[:80]}")
    print(f"    samples: {c['samples'][:5]}")

WA Ground Ambulance Locally Set Rates  (18 columns)
description: This dataset provides the locally established and contracted rates for local governmental ground ambulance service organ...

• Recorded Date  [calendar_date]
    desc:    Date the rate was submitted to the OIC
    samples: ['2025-07-14T00:00:00.000', '2024-09-18T00:00:00.000', '2024-09-19T00:00:00.000', '2024-09-23T00:00:00.000', '2024-09-24T00:00:00.000']
• GASO Name  [text]
    desc:    Ground ambulance service organization name
    samples: ['Douglas County Hospital District 2', 'City of Walla Walla Fire Department', 'City of Bremerton', 'Mukilteo Fire Department', 'Ferry County Emergency Medical Services District 1']
• GASO NPI  [text]
    desc:    Ground ambulance service organization 10-digit national provider identifier (NPI
    samples: ['1386699858', '1720004112', '1750354536', '1699886408', '1356476477']
• Service Counties  [text]
    desc:    The service counties for the submitting GASO
    samples: ['Douglas',